# Airbyte Python SDK Demo

This notebook demonstrates how to use the Airbyte Python SDK to interact with your Airbyte instance. We will cover:

1.  **Setup & Authentication**: Connecting to the Airbyte API.
2.  **Exploration**: Listing existing workspaces, sources, destinations, and connections.
3.  **Creating Resources**: Programmatically creating new resources.
4.  **Managing Connections**: Creating connections and triggering syncs.

## 1. Setup & Authentication

First, we import the necessary libraries and load our environment variables. Ensure you have a `.env` in the airbyte-client directory with `AIRBYTE_SERVER_URL`, `AIRBYTE_CLIENT_ID`, `AIRBYTE_CLIENT_SECRET`, `S3_ENDPOINT` and `DATABASE_HOST`.

In [ ]:
import airbyte_api
from airbyte_api import models, api
from dotenv import load_dotenv
import os
import requests

load_dotenv(dotenv_path="../airbyte-client/.env")

### Authenticate

If authentication has been enabled in the airbyte instance, we retrieve an access token using our client credentials and initialize the `AirbyteAPI` client.

In [11]:
authentication_required=False

if authentication_required==False:
    client = airbyte_api.AirbyteAPI(
        server_url=os.getenv("AIRBYTE_SERVER_URL")
    )
else:
    # Get the Bearer Token
    token_response = requests.post(
        f"{os.getenv('AIRBYTE_SERVER_URL')}/applications/token",
        json={
            "client_id": os.getenv("AIRBYTE_CLIENT_ID"),
            "client_secret": os.getenv("AIRBYTE_CLIENT_SECRET")
        }
    )

    if token_response.status_code == 200:
        token = token_response.json()["access_token"]
        print("Successfully authenticated.")
    else:
        print(f"Authentication failed: {token_response.status_code} - {token_response.text}")

    # Initialize the Client
    client = airbyte_api.AirbyteAPI(
        server_url=os.getenv("AIRBYTE_SERVER_URL"),
        security=models.Security(
                bearer_auth=token
            )
    )

## 2. Exploration

Let's explore what's currently in our Airbyte instance.

### List Workspaces

In [12]:
workspaces_response = client.workspaces.list_workspaces(request=api.ListWorkspacesRequest())
for workspace in workspaces_response.workspaces_response.data:
    print(f"Workspace: {workspace.name} (ID: {workspace.workspace_id})")

### List Sources

In [13]:
sources_response = client.sources.list_sources(request=api.ListSourcesRequest())
for source in sources_response.sources_response.data:
    print(f"Source: {source.name} (Type: {source.source_type}, ID: {source.source_id})")

### List Destinations

In [14]:
destinations_response = client.destinations.list_destinations(request=api.ListDestinationsRequest())
for destination in destinations_response.destinations_response.data:
    print(f"Destination: {destination.name} (Type: {destination.destination_type}, ID: {destination.destination_id})")

### List Connections

In [15]:
connections_response = client.connections.list_connections(request=api.ListConnectionsRequest())
for connection in connections_response.connections_response.data:
    print(f"Connection: {connection.name} (Status: {connection.status}, ID: {connection.connection_id})")

## 3. Creating Resources

Now we will create a new workspace, source, and destination programmatically.

### Create Workspace

In [ ]:
new_workspace_name = "python SDK Demo Workspace"

create_workspace_response = client.workspaces.create_workspace(
    request=models.WorkspaceCreateRequest(
        name=new_workspace_name
    )
)

new_workspace_id = create_workspace_response.workspace_response.workspace_id
print(f"Created Workspace: {new_workspace_name} (ID: {new_workspace_id})")

### Create Source (S3)

In [ ]:
create_source_response = client.sources.create_source(
    request=models.SourceCreateRequest(
        configuration=models.SourceS3(
            bucket="raw-data",
            streams=[
                models.SourceS3FileBasedStreamConfig(
                    format=models.SourceS3CSVFormat,
                    name='certificates',
                    globs=['epc/10-2025/certificates.csv'])
                ],
            aws_access_key_id="minioadmin",
            aws_secret_access_key="minioadmin",
            endpoint=os.getenv("S3_ENDPOINT")
            ),
        name="epc-sdk-source",
        workspace_id=new_workspace_id
    )
)

new_source_id = create_source_response.source_response.source_id
print(f"Created Source: epc-sdk-source (ID: {new_source_id})")

### Create Destination (Postgres)

In [ ]:
create_destination_response = client.destinations.create_destination(
    request=models.DestinationCreateRequest(
        configuration=models.DestinationPostgres(
            database="mydatabase",
            host=os.getenv("DATABASE_HOST"),
            port=5432,
            username="myuser",
            password="mypassword",
            schema="raw_data_sdk"
        ),
    name="postgres-sdk-destination",
    workspace_id=new_workspace_id
    )
)

new_destination_id = create_destination_response.destination_response.destination_id
print(f"Created Destination: postgres-sdk-destination (ID: {new_destination_id})")

## 4. Managing Connections

Finally, we connect the source to the destination and trigger a sync.

In [ ]:
try:
    create_connection_response = client.connections.create_connection(
        request=models.ConnectionCreateRequest(
            destination_id=new_destination_id,
            source_id=new_source_id,
            name='epc-connector-sdk',
        )
    )
    
    new_connection_id = create_connection_response.connection_response.connection_id
    print(f"Created Connection: epc-connector-sdk (ID: {new_connection_id})")
except Exception as e:
    print(f"Failed to create connection: {e}")

### Trigger Sync

In [ ]:
if 'new_connection_id' in locals():
    job_response = client.jobs.create_job(
        request=models.JobCreateRequest(
            connection_id=new_connection_id,
            job_type=models.JobTypeEnum.SYNC
        )
    )

    job_id = job_response.job_response.job_id
    print(f"Started Sync Job (ID: {job_id})")

### Check Job Status

In [ ]:
if 'job_id' in locals():
    get_job_response = client.jobs.get_job(
        request=api.GetJobRequest(
            job_id=job_id
        )
    )

    print(f"Job Status: {get_job_response.job_response.status}")

# 5. Cleanup/Teardown

### Delete connection

In [ ]:
delete_connection_response = client.connections.delete_connection(
    request=api.DeleteConnectionRequest(
        connection_id=new_connection_id
    )
)

### Delete destination

In [ ]:
delete_destination_response = client.destinations.delete_destination(
    request=api.DeleteDestinationRequest(
        destination_id=new_destination_id
    )
)

### Delete source

In [ ]:
delete_source_request = client.sources.delete_source(
    request=api.DeleteSourceRequest(
        source_id=new_source_id
    )
)

### Delete workspace

In [ ]:
delete_workspace_request = client.workspaces.delete_workspace(
    request=api.DeleteWorkspaceRequest(
        workspace_id=new_workspace_id)
)

### Delete all resources

In [ ]:
workspaces_response = client.workspaces.list_workspaces(request=api.ListWorkspacesRequest())
for workspace in workspaces_response.workspaces_response.data:
    print(f"Workspace: {workspace.name} (ID: {workspace.workspace_id})")
    
    delete_workspace_request = client.workspaces.delete_workspace(
        request=api.DeleteWorkspaceRequest(
            workspace_id=workspace.workspace_id)
    )
